In [0]:
# -----------------------------------------
# Healthcare Readmission Analytics Project
# Bronze Layer Ingestion
# -----------------------------------------

# Import Spark functions
from pyspark.sql.functions import *

# Path of raw dataset inside Unity Catalog Volume
source_path = "/Volumes/workspace/readmission_analytics/healthcare_catalog/diabetic_data.csv"

# Bronze table destination
bronze_table = "workspace.bronze.diabetic_data_bronze"

In [0]:
# Read raw CSV file

df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "true")
    .load(source_path)
)

display(df_raw)

In [0]:
# Display dataframe schema

df_raw.printSchema()

In [0]:
# Count rows and columns

row_count = df_raw.count()
column_count = len(df_raw.columns)

print(f"Total Rows: {row_count}")
print(f"Total Columns: {column_count}")

In [0]:
# Quick statistical summary

display(df_raw.describe())

In [0]:
# Preview key healthcare columns

display(
    df_raw.select(
        "race",
        "gender",
        "age",
        "time_in_hospital",
        "num_medications",
        "readmitted"
    )
)

In [0]:
# Add ingestion metadata columns

df_bronze = (
    df_raw
    .withColumn(
        "ingestion_timestamp",
        current_timestamp()
    )
    .withColumn(
        "source_file",
        col("_metadata.file_path")
    )
)

In [0]:
# Preview bronze dataframe

display(df_bronze)

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS workspace.bronze;

In [0]:
# Save Bronze dataframe as Delta Table

(
    df_bronze.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(bronze_table)
)

In [0]:
# Read Bronze table

df_bronze_check = spark.table(bronze_table)

display(df_bronze_check)

In [0]:
%sql
SELECT COUNT(*) AS total_records
FROM workspace.bronze.diabetic_data_bronze;